# Classificação


In [1]:
%%html
<link rel="stylesheet" href="./style.css">


In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import FunctionTransformer
from sklearn.tree import DecisionTreeClassifier

from data import (
    df3,
    df3_categorical_features,
    df3_input_features,
    df3_numerical_features,
    df3_target_feature,
)
from plotting_for_task import (
    display_cross_validation_results,
    display_cross_validation_summary,
    display_train_test_target_distribution,
)


In [3]:
seeds = [1, 2, 3, 4, 5]

categorical_features = df3_categorical_features
numerical_features = df3_numerical_features
input_features = df3_input_features
target_feature = df3_target_feature

input_data = df3[input_features].copy()
target_data = df3[target_feature].copy()


In [4]:
def encode_categorical_features(data_frame):
    """Encode categorical features as integer category codes."""
    encoded_data = data_frame.copy()

    for feature in categorical_features:
        encoded_data[feature] = encoded_data[feature].cat.codes

    return encoded_data


categorical_feature_indices = [
    input_features.index(feature) for feature in categorical_features
]


## Separação dos dados

Separamos os dados em **treino** (`80%`) e **teste** (`20%`) de forma estratificada pelo atributo objetivo:

- **Age group**: `10-19`, `20-29`, `30-39`, `40-49`, `50-59`, `60-69`, `70+`.


In [5]:
test_size = 0.2

(
    input_data_for_train,
    input_data_for_test,
    target_data_for_train,
    target_data_for_test,
) = train_test_split(
    input_data,
    target_data,
    test_size=test_size,
    random_state=seeds[0],
    stratify=target_data,
)

In [6]:
display_train_test_target_distribution(
    target_data_for_train,
    target_data_for_test,
);

Age Group,Train N,Test N,Total N,Train %,Test %
10-19,46,12,58,79.3%,20.7%
20-29,158,40,198,79.8%,20.2%
30-39,90,22,112,80.4%,19.6%
40-49,122,31,153,79.7%,20.3%
50-59,162,40,202,80.2%,19.8%
60-69,92,23,115,80.0%,20.0%
70+,69,17,86,80.2%,19.8%
General,739,185,924,80.0%,20.0%


## Decision tree


In [7]:
def _build_decision_tree_pipeline(
    random_state,
    use_smote,
):
    """Build a Decision Tree pipeline."""

    encoder = (
        "encoder",
        FunctionTransformer(
            encode_categorical_features,
            validate=False,
        ),
    )

    classifier = (
        "classifier",
        DecisionTreeClassifier(
            random_state=random_state,
        ),
    )

    if use_smote:
        smote = (
            "smote",
            SMOTENC(
                random_state=random_state,
                categorical_features=categorical_feature_indices,
            ),
        )

        return Pipeline(
            [
                encoder,
                smote,
                classifier,
            ]
        )

    return Pipeline(
        [
            encoder,
            classifier,
        ]
    )

### Balanceamento


In [8]:
def compare_smote(
    input_data,
    target_data,
    seeds,
    n_splits=5,
):
    """Compare Decision Tree performance with and without SMOTE."""

    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro",
    }

    results = []

    for current_seed in seeds:
        stratified_cross_validation = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=current_seed,
        )

        pipelines = {
            "Without SMOTE": _build_decision_tree_pipeline(
                random_state=current_seed,
                use_smote=False,
            ),
            "With SMOTE": _build_decision_tree_pipeline(
                random_state=current_seed,
                use_smote=True,
            ),
        }

        for strategy, pipeline in pipelines.items():
            cross_validation_results = cross_validate(
                pipeline,
                input_data,
                target_data,
                cv=stratified_cross_validation,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
                error_score="raise",
            )

            results.append(
                {
                    "Seed": current_seed,
                    "Strategy": strategy,
                    "Accuracy": cross_validation_results["test_accuracy"].mean(),
                    "Balanced Accuracy": cross_validation_results[
                        "test_balanced_accuracy"
                    ].mean(),
                    "Macro F1": cross_validation_results["test_f1_macro"].mean(),
                }
            )

    results = pd.DataFrame(results)

    comparison = results.groupby("Strategy").agg(
        {
            "Accuracy": ["mean", "std"],
            "Balanced Accuracy": ["mean", "std"],
            "Macro F1": ["mean", "std"],
        }
    )

    comparison.columns = [f"{metric} {stat}" for metric, stat in comparison.columns]

    comparison = comparison.reset_index()

    metrics = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
    ]

    for metric in metrics:
        comparison[metric] = (
            comparison[f"{metric} mean"].map(lambda value: f"{value:.3f}")
            + " ± "
            + comparison[f"{metric} std"].map(lambda value: f"{value:.3f}")
        )

    return comparison, results

In [9]:
smote_comparison, smote_results = compare_smote(
    input_data_for_train,
    target_data_for_train,
    seeds,
)

metric_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
]

display_cross_validation_results(
    smote_results,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Cross-Validation Results",
)

display_cross_validation_summary(
    smote_comparison,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Comparison",
)

Seed,Strategy,Accuracy,Balanced Accuracy,Macro F1
1,Without SMOTE,0.346,0.336,0.324
1,With SMOTE,0.355,0.357,0.342
2,Without SMOTE,0.376,0.367,0.359
2,With SMOTE,0.399,0.400,0.388
3,Without SMOTE,0.394,0.377,0.367
3,With SMOTE,0.405,0.411,0.393
4,Without SMOTE,0.367,0.342,0.333
4,With SMOTE,0.375,0.354,0.346
5,Without SMOTE,0.357,0.353,0.339
5,With SMOTE,0.353,0.374,0.347


Strategy,Accuracy,Balanced Accuracy,Macro F1
With SMOTE,0.377 ± 0.024,0.379 ± 0.025,0.363 ± 0.025
Without SMOTE,0.368 ± 0.018,0.355 ± 0.017,0.345 ± 0.018
